# Step 8: Your first web server

![Step 8 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/08-step.png)

**Goal:** run a real web server on your computer and visit it — from a browser, from Python, and from your phone.

**Concept:** a server is a program that *waits* for questions and answers them. Until now we've always been the one *asking* — today we switch chairs.

## Step 8.0 — Setup

Nothing to configure this time — the whole lesson runs inside the notebook. No secrets, no folders, no `.env` needed.

## Step 8.1 — You've already met a server

Think back to lesson 2. The IP Webcam app on the phone gave us a URL:

```
http://PHONE_IP:8080/photo.jpg
```

That worked because the phone was running a **web server** — a program sitting there waiting for anyone on the WiFi to ask it something. Our computer was the **client** — the one asking.

Client asks, server answers. Same dance every time. Today we build the *other side*: our own server, on our own computer.

## Step 8.2 — Hello world in 10 lines

FastAPI is the easiest Python web framework. An `app` object, a `@app.get("/")` decorator, a function that returns a dict — FastAPI turns the dict into JSON automatically.

Each `@app.get(...)` line registers a **route**: "when someone asks for this path, run this function." 

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Bird Watcher", version="0.0.0")


@app.get("/")
def index() -> dict:
    """Landing page."""
    return {"app": "bird-watcher", "status": "watching"}


@app.get("/health")
def health() -> dict:
    """Health check — useful for uptime monitoring."""
    return {"status": "ok"}

## Step 8.3 — Run it for real

A server's whole job is to *wait forever* — so if we started it normally, this notebook would freeze on this cell for the rest of time. The trick: run it on a **background thread**, a helper that waits so we don't have to.

**Run this cell once.** If you ever see `address already in use`, an old server is still running — jump ahead to the stop cell, then come back.

In [ ]:
import threading
import time

import uvicorn

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8123, log_level="info"))

threading.Thread(target=server.run, daemon=True).start()
time.sleep(1)  # give the server a second to boot

print("Your server is LIVE.")
print("Open  http://localhost:8123  in your browser.")

## Step 8.4 — Visit your server

Three clients, one server. Try them all:

**1. Your browser.** Open [http://localhost:8123](http://localhost:8123) — there's your dict, as JSON, on a real web page. Then try [http://localhost:8123/docs](http://localhost:8123/docs): FastAPI auto-generates a control panel for every route you defined. You wrote 10 lines and got a website *and* documentation.

Every time you refresh, the server logs the request (`GET / 200`) — the log lines show up in the notebook output, usually attached to the most recently run cell. That's the server *noticing you*.

**2. Python.** The same `requests` library from lesson 2 — because Python, browsers, and phones are all just clients:

In [ ]:
import requests

r = requests.get("http://localhost:8123")
print(r.status_code, r.json())

r = requests.get("http://localhost:8123/health")
print(r.status_code, r.json())

**3. Your phone.** This is why we said `host="0.0.0.0"` — it means "answer *anyone on the WiFi*, not just this computer." (Your OS may pop up a firewall question — allow it.)

Run the next cell to find your computer's address, then type that URL into the phone's browser.

In lesson 2, the phone served and the computer asked. **Look who's serving now.**

In [ ]:
import socket

s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
s.connect(("8.8.8.8", 80))  # doesn't send anything — just picks the right network card
MY_IP = s.getsockname()[0]
s.close()

print(f"On your phone's browser, open:  http://{MY_IP}:8123")

## Step 8.5 — Stop the server

Servers run until told to stop — that's the whole job. Tell it to stop:

In [ ]:
server.should_exit = True
time.sleep(1)

print("Server stopped. Refresh the browser — nobody answers anymore.")

## Step 8.6 — TestClient: a pretend browser for tests

Starting a real server just to check your code works is overkill — it grabs a port, needs WiFi, has to be stopped. `TestClient` is a *pretend browser*: it asks your app questions directly, no server required. Same questions, same answers, zero ceremony — perfect for acceptance checks like this one:

In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)
r = client.get("/")
print(r.status_code, r.json())
assert r.status_code == 200
assert r.json() == {"app": "bird-watcher", "status": "watching"}

r = client.get("/health")
print(r.status_code, r.json())
assert r.status_code == 200
assert r.json() == {"status": "ok"}
print("✅ / and /health both respond")

## Step 8.7 — Why a factory function?

For a real app, you want to *configure* it at startup — where do the sightings come from? A factory takes that as an argument and returns an `app`:

```python
def create_app(storage: SightingStorage) -> FastAPI:
    app = FastAPI(...)
    # ... add routes that ask `storage` for sightings ...
    return app
```

Notice what the factory asks for: not a database file, but the `SightingStorage` contract from lesson 6. The routes will only ever talk to the contract — so any store that keeps the promise works, SQLite or otherwise.

We'll wire that up next, where the factory gets the `/gallery` route — and you'll click through *your own bird gallery* in the browser.

## What's next

**Step 9:** open [09-gallery.ipynb](09-gallery.ipynb) — we'll define the `create_app` factory, add the `/gallery` route, and serve each sighting's photo straight from the database.